# 01 · Data Preparation

## 0 · Setup 

In [ ]:
from pathlib import Path
import sys, yaml

# Resolve project root when notebook is launched from either project root or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise FileNotFoundError(f'Cannot find project root from {Path.cwd()}')

sys.path.insert(0, str(PROJECT_ROOT))

# Optional dependency check (install via terminal if missing)
missing = []
for pkg in ['pretty_midi', 'yaml']:
    try:
        __import__(pkg)
    except Exception:
        missing.append(pkg)
if missing:
    print(f'Missing packages: {missing}')
    print('Install with: pip install pretty_midi PyYAML')

with open(PROJECT_ROOT / 'configs' / 'base.yaml') as f:
    config = yaml.safe_load(f)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('Config loaded:', config)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from pathlib import Path

from src.data.augment import transpose, temporal_jitter, augment
from src.data.dataset import MusicDataset

DATA_DIR   = Path(PROJECT_ROOT) / 'data' / 'processed'
GENRES     = ['jazz', 'pop']
SEQ_LEN    = config['data']['seq_len']          
INPUT_DIM  = config['model']['input_dim']       
BATCH_SIZE = config['training']['batch_size']   

print(f'DATA_DIR = {DATA_DIR}')
print(f'SEQ_LEN={SEQ_LEN}  INPUT_DIM={INPUT_DIM}  BATCH_SIZE={BATCH_SIZE}')

---
## 1 · Verify `.npy` File Shapes

In [ ]:
splits = ['train', 'val', 'test']
arrays = {}

for genre in GENRES:
    arrays[genre] = {}
    for split in splits:
        fpath = DATA_DIR / genre / f'{split}.npy'
        assert fpath.exists(), f'Missing: {fpath}'
        arr = np.load(str(fpath))
        arrays[genre][split] = arr
        density = float(arr[:, :, :88].mean())
        print(f'  {genre:<12} {split:<6}  shape={str(arr.shape):<22}  '
              f'dtype={arr.dtype}  density={density:.4f}')

for genre in GENRES:
    for split in splits:
        arr = arrays[genre][split]
        assert arr.ndim == 3
        assert arr.shape[1] == SEQ_LEN,   f'{genre}/{split}: seq_len mismatch'
        assert arr.shape[2] == INPUT_DIM, f'{genre}/{split}: input_dim mismatch'
        assert arr.dtype == np.float32,   f'{genre}/{split}: dtype must be float32'

print('\nAll shape assertions passed ✓')

---
## 2 · Sample Piano Rolls (one per genre)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
rng = np.random.default_rng(0)

for ax, genre in zip(axes, GENRES):
    arr = arrays[genre]['train']
    seg = arr[rng.integers(0, len(arr))]   
    ax.imshow(seg[:, :88].T, aspect='auto', origin='lower',
              cmap='Blues', interpolation='nearest')
    ax.set_title(genre.capitalize(), fontsize=11)
    ax.set_xlabel('Time step')

axes[0].set_ylabel('Pitch (MIDI 21-108)')
fig.suptitle('Sample Piano Rolls - train split', fontsize=12)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/outputs/figures/sample_piano_rolls.png',
            bbox_inches='tight', dpi=120)
plt.show()

---
## 3 · Augmentation Sanity Check  (`src/data/augment.py`)

In [ ]:
rng  = np.random.default_rng(1)
arr  = arrays['classical']['train']
idx  = rng.integers(0, len(arr))
orig = arr[idx, :, :88].copy()  

trans     = transpose(orig, +5)
jittered  = temporal_jitter(orig, drop_p=0.05)
augmented = augment(orig.copy())  

fig, axes = plt.subplots(1, 4, figsize=(16, 3), sharey=True)
for ax, roll, title in zip(
    axes,
    [orig, trans, jittered, augmented],
    ['Original', 'transpose(+5)', 'temporal_jitter', 'augment() full']
):
    ax.imshow(roll.T, aspect='auto', origin='lower',
              cmap='Blues', interpolation='nearest')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Time step')

axes[0].set_ylabel('Pitch')
fig.suptitle(f'src/data/augment.py - Classical sample #{idx}', fontsize=12)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/outputs/figures/augmentation_check.png',
            bbox_inches='tight', dpi=120)
plt.show()

assert trans[:, :5].sum() == 0, 'transpose boundary leak!'
print('transpose boundary check ✓')

---
## 4 · Dataset & DataLoader Check  (`src/data/dataset.py`)

In [ ]:
train_ds = MusicDataset(str(DATA_DIR), GENRES, split='train')
val_ds   = MusicDataset(str(DATA_DIR), GENRES, split='val')
test_ds  = MusicDataset(str(DATA_DIR), GENRES, split='test')

for ds, name in [(train_ds, 'train'), (val_ds, 'val'), (test_ds, 'test')]:
    print(f'  {name:<6}  total={len(ds):>6}  per genre: {ds.genre_counts()}')

assert train_ds.do_augment == True,  'train should augment'
assert val_ds.do_augment   == False, 'val should NOT augment'
assert test_ds.do_augment  == False, 'test should NOT augment'
print('Augmentation flags ✓')

In [ ]:
x, rhythm_gt, label = train_ds[0]

assert x.shape         == (SEQ_LEN, INPUT_DIM)
assert rhythm_gt.shape == (SEQ_LEN,)
assert label.dtype     == torch.int64
assert set(rhythm_gt.tolist()).issubset({0.0, 1.0}), 'rhythm_gt must be binary'

print(f'x          : {tuple(x.shape)}   (expected ({SEQ_LEN}, {INPUT_DIM}))')
print(f'rhythm_gt  : {tuple(rhythm_gt.shape)}      (expected ({SEQ_LEN},))')
print(f'label      : {label.item()}                    (expected 0/1/2)')
print(f'onset rate : {rhythm_gt.float().mean():.2%}')
print('Shape & dtype assertions ✓')

In [ ]:
loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=2, pin_memory=True)

batch_x, batch_rhythm, batch_label = next(iter(loader))
print(f'Batch x          : {tuple(batch_x.shape)}')
print(f'Batch rhythm_gt  : {tuple(batch_rhythm.shape)}')
print(f'Batch label      : {tuple(batch_label.shape)}')
print(f'Label distribution: { {i: int((batch_label==i).sum()) for i in range(3)} }')

---
## 5 · Rhythm Ground Truth Visualisation

In [ ]:
jazz_offset = sum(len(arrays[g]['val']) for g in GENRES[:GENRES.index('jazz')])
x, rhythm_gt, label = val_ds[jazz_offset + 5]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 4), sharex=True)

ax1.imshow(x[:, :88].numpy().T, aspect='auto', origin='lower',
           cmap='Blues', interpolation='nearest')
ax1.set_ylabel('Pitch')
ax1.set_title('Jazz piano roll (val, no augmentation)')

ax2.bar(range(SEQ_LEN), rhythm_gt.numpy(), color='steelblue', width=0.8)
ax2.set_ylabel('Onset')
ax2.set_xlabel('Time step')
ax2.set_ylim(-0.1, 1.2)
ax2.set_yticks([0, 1])
ax2.set_title('Rhythm ground truth  [dataset.py __getitem__]')

plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/outputs/figures/rhythm_gt_example.png',
            bbox_inches='tight', dpi=120)
plt.show()

---
## 6 · Summary

In [ ]:
total = len(train_ds) + len(val_ds) + len(test_ds)
print('=' * 55)
print('Data Preparation Summary')
print('=' * 55)
print(f'  train : {len(train_ds):>7}')
print(f'  val   : {len(val_ds):>7}')
print(f'  test  : {len(test_ds):>7}')
print(f'  total : {total:>7}')
print(f'  disk  : ~{total * SEQ_LEN * INPUT_DIM * 4 / 1e6:.0f} MB')
print('  augmentation : on-the-fly in src/data/dataset.py (train only)')
print('=' * 55)
print('\nAll checks passed. Proceed to 02_train.ipynb')